# Dataset cleanup

Walks every `Patch_<size>/` directory under `ROOT` (recursively) and
identifies files to delete.

**Keep rules** (case-insensitive):

- Any file with extension `.tif`, `.tiff`, `.png`
- Any `.parquet` file sitting directly at the root of its `Patch_<size>` dir
  (e.g. `Patch_128/metadata_128.parquet` is kept; a `.parquet` nested in a subfolder is not)

**Everything else is flagged for deletion.**

**Workflow:**

1. Run cells 1–3 to scan and list the deletion candidates (dry run only).
2. Review the list.
3. Set `CONFIRM_DELETE = True` in cell 4 and run it to actually delete.

Files sitting outside any `Patch_<size>/` directory are reported as orphans
and left untouched — review them manually.

Directories are never deleted, even if they become empty.


In [1]:
# =====================================================
# Configuration
# =====================================================
import os
import re
from pathlib import Path
from collections import defaultdict

# -----------------------------------------------------
# ROOT containing Patch_<size>/ subdirectories
# -----------------------------------------------------
ROOT = Path("/home/ubuntu/SENSERO/GeoTiff")

# Extensions to keep (lowercase, with leading dot)
KEEP_EXTS = {".tif", ".tiff", ".png"}

# Regex to extract size from a directory named "Patch_<size>" (case-insensitive)
PATCH_DIR_RE = re.compile(r"^Patch_(\d+)$", re.IGNORECASE)

assert ROOT.exists() and ROOT.is_dir(), f"ROOT not found: {ROOT}"
print(f"Root: {ROOT}")

Root: /home/ubuntu/SENSERO/GeoTiff


In [2]:
# =====================================================
# Scan: identify Patch_<size>/ dirs and walk each recursively
# =====================================================
patch_dirs = []           # list[(size, Path)]
for entry in sorted(ROOT.iterdir()):
    if entry.is_dir():
        m = PATCH_DIR_RE.match(entry.name)
        if m:
            patch_dirs.append((int(m.group(1)), entry))

if not patch_dirs:
    raise RuntimeError(f"No Patch_<size>/ directories found under {ROOT}")

print(f"Found {len(patch_dirs)} patch directories:")
for size, d in patch_dirs:
    print(f"  Patch_{size:>3d}  ->  {d}")

# Per-size deletion candidates and kept counts
to_delete = []                          # list[Path]
keep_counts = defaultdict(int)          # size -> n_kept
delete_counts = defaultdict(int)        # size -> n_to_delete

for size, pdir in patch_dirs:

    for fp in pdir.rglob("*"):
        if not fp.is_file():
            continue

        ext = fp.suffix.lower()
        name = fp.name.lower()

        # Keep rule 1: TIFF / PNG anywhere inside Patch_<size>/
        if ext in KEEP_EXTS:
            keep_counts[size] += 1
            continue

        # Keep rule 2: any .parquet directly at the root of Patch_<size>/
        if ext == ".parquet" and fp.parent == pdir:
            keep_counts[size] += 1
            continue

        # Everything else: delete
        to_delete.append(fp)
        delete_counts[size] += 1

# -----------------------------------------------------
# Orphans: files outside any Patch_<size>/ subtree
# -----------------------------------------------------
patch_dir_paths = {d.resolve() for _, d in patch_dirs}

def inside_any_patch_dir(p):
    p = p.resolve()
    for parent in (p, *p.parents):
        if parent in patch_dir_paths:
            return True
    return False

orphans = []
for fp in ROOT.rglob("*"):
    if fp.is_file() and not inside_any_patch_dir(fp):
        orphans.append(fp)

# -----------------------------------------------------
# Summary
# -----------------------------------------------------
print("\nPer-size summary:")
print(f"  {'size':>6s}  {'keep':>10s}  {'delete':>10s}")
for size, _ in patch_dirs:
    print(f"  {size:>6d}  {keep_counts[size]:>10d}  {delete_counts[size]:>10d}")

print(f"\nTotal files to keep   : {sum(keep_counts.values()):,}")
print(f"Total files to delete : {len(to_delete):,}")
print(f"Orphan files (outside Patch_<size>/): {len(orphans)}")

Found 8 patch directories:
  Patch_112  ->  /home/ubuntu/SENSERO/GeoTiff/Patch_112
  Patch_120  ->  /home/ubuntu/SENSERO/GeoTiff/Patch_120
  Patch_128  ->  /home/ubuntu/SENSERO/GeoTiff/Patch_128
  Patch_224  ->  /home/ubuntu/SENSERO/GeoTiff/Patch_224
  Patch_256  ->  /home/ubuntu/SENSERO/GeoTiff/Patch_256
  Patch_280  ->  /home/ubuntu/SENSERO/GeoTiff/Patch_280
  Patch_336  ->  /home/ubuntu/SENSERO/GeoTiff/Patch_336
  Patch_ 64  ->  /home/ubuntu/SENSERO/GeoTiff/Patch_64

Per-size summary:
    size        keep      delete
     112       50007           2
     120       50007           2
     128       50007           2
     224       50007           2
     256       50007           2
     280       50008           3
     336       50007           2
      64       50007           2

Total files to keep   : 400,057
Total files to delete : 17
Orphan files (outside Patch_<size>/): 13


In [3]:
# =====================================================
# List deletion candidates
# =====================================================
# Group by extension/name pattern for an easier eyeball check
from collections import Counter

if not to_delete:
    print("Nothing to delete.")
else:
    # Breakdown by extension
    ext_counter = Counter(fp.suffix.lower() or "<noext>" for fp in to_delete)
    print("Breakdown by extension:")
    for ext, n in sorted(ext_counter.items(), key=lambda x: -x[1]):
        print(f"  {ext:>12s}  {n:>8d}")

    # Sample (up to 50 per size to avoid wall-of-text)
    SAMPLE = 50
    by_size = defaultdict(list)
    for fp in to_delete:
        # Find the Patch_<size> dir this file belongs to
        for size, pdir in patch_dirs:
            try:
                fp.resolve().relative_to(pdir.resolve())
                by_size[size].append(fp)
                break
            except ValueError:
                continue

    print(f"\nDeletion candidates (showing up to {SAMPLE} per patch size):")
    for size in sorted(by_size):
        files = by_size[size]
        print(f"\n--- Patch_{size} ({len(files)} files) ---")
        for fp in files[:SAMPLE]:
            print(f"  {fp.relative_to(ROOT)}")
        if len(files) > SAMPLE:
            print(f"  ... and {len(files) - SAMPLE} more")

# Orphans (always show all — there shouldn't be many)
if orphans:
    print(f"\n--- Orphans outside Patch_<size>/ (NOT deleted) ---")
    for fp in orphans:
        print(f"  {fp.relative_to(ROOT)}")

Breakdown by extension:
          .csv        17

Deletion candidates (showing up to 50 per patch size):

--- Patch_64 (2 files) ---
  Patch_64/split_summary.csv
  Patch_64/metadata_64.csv

--- Patch_112 (2 files) ---
  Patch_112/split_summary.csv
  Patch_112/metadata_112.csv

--- Patch_120 (2 files) ---
  Patch_120/metadata_120.csv
  Patch_120/split_summary.csv

--- Patch_128 (2 files) ---
  Patch_128/metadata_128.csv
  Patch_128/split_summary.csv

--- Patch_224 (2 files) ---
  Patch_224/metadata_224.csv
  Patch_224/split_summary.csv

--- Patch_256 (2 files) ---
  Patch_256/split_summary.csv
  Patch_256/metadata_256.csv

--- Patch_280 (3 files) ---
  Patch_280/metadata_280.csv
  Patch_280/split_summary.csv
  Patch_280/split_summary_buffered.csv

--- Patch_336 (2 files) ---
  Patch_336/split_summary.csv
  Patch_336/metadata_336.csv

--- Orphans outside Patch_<size>/ (NOT deleted) ---
  swap_log.csv
  split_summary.parquet
  leakage_swap_log.csv
  canonical_checkpoint.parquet
  leakage_

In [4]:
# =====================================================
# DELETE (dry run by default)
# =====================================================
# Review the list printed above. When ready to actually delete,
# set CONFIRM_DELETE = True and re-run this cell.

CONFIRM_DELETE = True

if not CONFIRM_DELETE:
    print(f"[DRY RUN] Would delete {len(to_delete):,} files.")
    print("Set CONFIRM_DELETE = True in this cell and re-run to actually delete.")
else:
    print(f"Deleting {len(to_delete):,} files...")
    n_deleted = 0
    n_failed = 0
    failures = []
    for fp in to_delete:
        try:
            fp.unlink()
            n_deleted += 1
        except OSError as e:
            n_failed += 1
            failures.append((fp, str(e)))
    print(f"Deleted: {n_deleted:,}")
    if n_failed:
        print(f"Failed : {n_failed:,}")
        for fp, err in failures[:20]:
            print(f"  {fp}: {err}")
        if len(failures) > 20:
            print(f"  ... and {len(failures) - 20} more")

Deleting 17 files...
Deleted: 17
